# Dense overflow

The overflow benchmark of Ilıcak, Adcroft, Griffies and Hallberg (2012): a 20 km plug of 10 °C water is
released on a 500 m shelf into a 20 °C basin 2000 m deep and descends the slope.

The configuration carries no tracer diffusivity and no closure, so the whole rise of the reference potential
energy is spurious and the effective diapycnal diffusivity read off it measures the discretization alone.

Run the sweep with `dense_overflow/test.jl` before executing this notebook.

In [ ]:
using Pkg
Pkg.activate("../")
using Suppressor
@suppress begin
    using Oceananigans, CairoMakie, TimestepperTestCases, Statistics
    using LaTeXStrings
end

In [ ]:
# Loader path: folder * "dense_overflow_" * label * "_" * timestepper * ".jld2", the name
# `dense_overflow(d::Discretization)` writes. Derived from the package so the notebook cannot drift
# from the runner.
folder = "../dense_overflow/"

variants = [(; label = d.label, timestepper = string(d.timestepper)) for d in discretizations()]

available = filter(v -> isfile(folder * "dense_overflow_" * v.label * "_" * v.timestepper * ".jld2"), variants)
labels    = [v.label for v in available]
styles    = [plot_style(v.label) for v in available]

cases = [load_dense_overflow(folder, v.label, v.timestepper) for v in available]

labels

## The plume

Temperature at the end of the run. The dense tongue should reach the basin floor still carrying its shelf
temperature; a scheme that mixes it away on the way down arrives warmer and shallower.

In [ ]:
ticks(vec) = (vec, latexstring.(string.(vec)))

# the tracers are written volume weighted, and the dry cells are written as zeros
function temperature(case, t = length(case[:times]))
    T = interior(compute!(Field(case[:T][t] / case[:VCCC][t])), :, 1, :)
    return replace(T, 0.0 => NaN)
end

grid = cases[1][:T].grid
x = grid.xᶜᵃᵃ[1:size(grid, 1)] ./ 1e3
z = grid.z.cᵃᵃᶜ[1:size(grid, 3)]

fig = Figure(size = (330 * min(length(cases), 3), 250 * cld(length(cases), 3)))

hm = nothing
for (n, case) in enumerate(cases)
    row, col = fldmod1(n, 3)
    ax = Axis(fig[row, col]; title = labels[n],
              xlabel = row == cld(length(cases), 3) ? L"\text{x [km]}" : "",
              ylabel = col == 1 ? L"\text{Depth [m]}" : "",
              xticks = ticks([0, 50, 100, 150, 200]), yticks = ticks([-2000, -1000, 0]))
    global hm = heatmap!(ax, x, z, temperature(case); colorrange = (10, 20), colormap = :thermal)
end

Colorbar(fig[:, end+1], hm; label = L"T \text{ [}^\circ\text{C]}")
current_figure()

## Reference potential energy

`RPE` is the potential energy of the adiabatically re-sorted state, so it can only rise through transport
across density classes. With no explicit diffusivity every increment is numerical.

In [ ]:
fig = Figure(size = (750, 320))
ax  = Axis(fig[1, 1]; xlabel = L"\text{time [hours]}",
           ylabel = L"\text{RPE} - \text{RPE}(0) \text{ [m}^2\text{s}^{-2}\text{]}")

for (n, case) in enumerate(cases)
    lines!(ax, case[:times] ./ 3600, case[:rpe] .- case[:rpe][1];
           label = labels[n], styles[n]...)
end

Legend(fig[1, 2], ax)
current_figure()

## Spurious diapycnal diffusivity

`κ = (dRPE/dt) / N²`, the slope of the curves above divided by the equivalent stratification of the
two-layer front. This is the number the case exists to report.

In [ ]:
κ = [dense_overflow_diffusivity(case) for case in cases]

reference = findfirst(==("AB2-SE"), labels)

println(rpad("case", 16), rpad("κ_num [m² s⁻¹]", 20), "relative to AB2-SE")
for (n, label) in enumerate(labels)
    relative = isnothing(reference) ? NaN : κ[n] / κ[reference]
    println(rpad(label, 16), rpad(round(κ[n], sigdigits = 4), 20), round(relative, digits = 3))
end

In [ ]:
fig = Figure(size = (620, 320))
ax  = Axis(fig[1, 1]; ylabel = L"\kappa_{num} \text{ [m}^2\text{s}^{-1}\text{]}",
           xticks = (eachindex(labels), labels), xticklabelrotation = π/6)

barplot!(ax, eachindex(labels), κ; color = [s.color for s in styles])
current_figure()